# Phase 5 — le juge (modèle de langage ouvert)

**Agent d'audit de conformité documentaire**

Pour chaque cas (une clause + une règle de contrôle), un petit modèle de langage ouvert doit répondre en JSON :
`conforme`, `non_conforme` ou `a_verifier`, avec une justification et les sources qu'il cite.

Deux modèles sont comparés : **Qwen2.5-1.5B-Instruct** et **Qwen2.5-3B-Instruct** (fp16, sans quantification, donc sans
risque d'incompatibilité). Deux conditions : **avec la règle** fournie dans le prompt, et **sans règle** (même modèle, jugeant
de mémoire), pour mesurer ce qu'apporte la règle. Décodage glouton : mêmes entrées, mêmes sorties.

## Règle importante

**Ce notebook n'affiche aucun verdict, et vous ne devez lire aucune réponse du modèle avant d'avoir fini d'annoter la feuille
Excel.** Sinon, vos annotations seraient influencées par le modèle. Le notebook sauvegarde les réponses brutes ; la mesure se fait
ensuite sur votre PC avec `scripts/eval_judge.py`.

## Mode d'emploi

1. **Exécution** > **Modifier le type d'exécution** > **T4 GPU** > Enregistrer
2. **Exécution** > **Tout exécuter**
3. À l'envoi des fichiers, déposez **DEUX** fichiers : `data\judge\cases.jsonl` et `src\judge\prompt.py`

> Si Colab **redémarre tout seul** après la cellule d'installation, c'est normal : relancez à partir de la cellule suivante.

Durée : environ **25 à 40 minutes** (téléchargement des modèles compris). Les réponses sont sauvegardées au fur et à mesure :
si la session s'interrompt, relancez simplement le notebook, il reprend là où il s'est arrêté.

Licences : Qwen2.5-1.5B est sous licence Apache 2.0 ; Qwen2.5-3B sous licence de recherche Qwen (usage non commercial,
adapté à un projet universitaire).

## 1. GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU détecté :", torch.cuda.get_device_name(0))
else:
    print("AUCUN GPU. Exécution > Modifier le type d'exécution > T4 GPU, puis relance.")

## 2. Installation

In [ ]:
import sys
print("Python", sys.version.split()[0])
!pip install "transformers==4.46.3" "tokenizers==0.20.3" "huggingface-hub<1.0" "accelerate>=1.1,<2" scikit-learn 2>&1 | tail -6

In [ ]:
# Vérifie que la bonne version est INSTALLÉE et CHARGÉE (Colab fournit transformers 5, qui est incompatible
# avec ce notebook : une version déjà chargée en mémoire masquerait l'installation).
import sys, os, importlib.metadata as md

on_disk = md.version("transformers")
loaded = sys.modules["transformers"].__version__ if "transformers" in sys.modules else None
print(f"transformers installé : {on_disk}  |  chargé en mémoire : {loaded}")
if on_disk != "4.46.3":
    raise SystemExit("La version 4.46.3 n'a pas pu être installée : copie l'affichage de la cellule précédente.")
if loaded not in (None, "4.46.3"):
    print("\nLa session utilisait une autre version : redémarrage automatique (normal).")
    print("-> Quand Colab a redémarré, relance à partir de la CELLULE 3 (pas besoin de refaire celle-ci).")
    os.kill(os.getpid(), 9)

# Test d'import immédiat : s'il y a un problème, on le voit ici en 5 secondes, pas après 25 minutes.
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback)
print("Imports OK : transformers", __import__("transformers").__version__)

## 3. Dépôt des fichiers

Déposez `cases.jsonl` **et** `prompt.py`.

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
for name in ("cases.jsonl", "prompt.py"):
    print(("OK       " if os.path.exists(name) else "MANQUANT ") + name)
assert os.path.exists("cases.jsonl") and os.path.exists("prompt.py"), "Il manque un fichier : relance cette cellule."

## 4. Chargement des cas

In [ ]:
import json
from collections import Counter

cases = [json.loads(line) for line in open("cases.jsonl", encoding="utf-8")]
from prompt import build_messages        # le même prompt que celui testé sur le PC

print(f"{len(cases)} cas :", dict(Counter(c["family"] for c in cases)))
m = build_messages(cases[0], with_rule=True)
print(f"exemple de prompt : {len(m[1]['content'])} caractères")      # on n'affiche pas le contenu

## 5. Configuration

In [ ]:
MODELS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
MAX_NEW_TOKENS = 200
OUT_FILE = "judge_results.jsonl"
print(f"{len(MODELS)} modèles | décodage glouton | {MAX_NEW_TOKENS} jetons maximum par réponse")

## 6. Chargement d'un modèle et génération

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def make_generator(model_name):
    """Retourne (generate, cleanup) : generate(messages) -> texte de la réponse."""
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, low_cpu_mem_usage=True).to("cuda").eval()

    def generate(messages):
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 temperature=None, top_p=None, top_k=None, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    def cleanup():
        nonlocal model
        del model
        gc.collect()
        torch.cuda.empty_cache()

    return generate, cleanup

## 7. Exécution (reprise automatique)

Avec la règle : tous les cas. Sans la règle : seulement les cas annotés par l'équipe.

In [ ]:
import json
import os
import time


def run_all():
    done = set()
    if os.path.exists(OUT_FILE):
        for line in open(OUT_FILE, encoding="utf-8"):
            r = json.loads(line)
            done.add((r["model"], r["condition"], r["case_id"]))
    jobs = [(c, "with_rule") for c in cases] + [(c, "without_rule") for c in cases if c["a_annoter"]]
    for model_name in MODELS:
        todo = [(c, cond) for c, cond in jobs if (model_name, cond, c["id"]) not in done]
        print(f"\n{model_name} : {len(todo)} réponses à produire ({len(jobs) - len(todo)} déjà faites)")
        if not todo:
            continue
        generate, cleanup = make_generator(model_name)
        t0 = time.time()
        with open(OUT_FILE, "a", encoding="utf-8") as f:
            for i, (c, cond) in enumerate(todo, 1):
                raw = generate(build_messages(c, with_rule=(cond == "with_rule")))
                f.write(json.dumps({"case_id": c["id"], "model": model_name, "condition": cond, "raw": raw}, ensure_ascii=False) + "\n")
                f.flush()
                if i % 20 == 0 or i == len(todo):
                    print(f"  {i}/{len(todo)}  ({time.time() - t0:.0f} s)")
        cleanup()


run_all()

## 8. Comptage (aucun verdict affiché)

In [ ]:
import json
from collections import Counter

rows = [json.loads(line) for line in open(OUT_FILE, encoding="utf-8")]
print(f"{len(rows)} réponses enregistrées dans {OUT_FILE}\n")
for key, n in sorted(Counter((r["model"], r["condition"]) for r in rows).items()):
    sel = [r["raw"] for r in rows if (r["model"], r["condition"]) == key]
    json_like = sum(1 for t in sel if "{" in t and "}" in t)
    print(f"{key[0]:28s} {key[1]:13s} {n:>4} réponses | contiennent un JSON : {json_like}/{n} | longueur moyenne {sum(map(len, sel)) // max(n, 1)} car.")

## 9. Téléchargement

Placez le fichier téléchargé dans `data\judge\` de votre projet sous le nom `judge_results.jsonl`.

In [ ]:
from google.colab import files
files.download("judge_results.jsonl")